# Colab — Entrenar SVM χ² (pipeline clásico)

Lee `classical_features.npz` desde Drive, entrena la SVM con calidad completa (`sample_steps=2`, `n_jobs=-1` — aprovecha la RAM y cores de Colab), guarda el clasificador y lo sube a Drive.

Suele tardar **5-15 min**. No re-corre Selective Search / SIFT / HOG — eso ya está en el .npz.

## Prerequisitos

En Drive `/MyDrive/grocery-detection/processed/`:
- `classical_features.npz`  (output de `colab_build_features.ipynb`)

## 1. Bootstrap

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import mount_drive, install_deps, sync_from_drive, sync_to_drive, run_script
print("helpers cargados")

In [ ]:
mount_drive()
install_deps()

## 2. Bajar features.npz desde Drive

In [ ]:
sync_from_drive(["classical_features.npz"])

## 3. Entrenar SVM (calidad full)

`sample_steps=2` da output dim = D × 3 (mejor que `=1` en calidad).  `n_jobs=-1` aprovecha los cores de Colab — aquí no hay riesgo de OOM como en un Mac de 8 GB.

In [ ]:
import time, pickle
import numpy as np
import sklearn
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

print(f"sklearn: {sklearn.__version__}")

data = np.load("data/processed/classical_features.npz")
X, y = data["X"], data["y"]
print(f"X: {X.shape}  y: {y.shape}")
unique, counts = np.unique(y, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  class {c}: {n}")

X = np.clip(X.astype(np.float32, copy=False), 0.0, None)

t0 = time.time()
print("\nStep 1: AdditiveChi2Sampler.fit_transform...")
sampler = AdditiveChi2Sampler(sample_steps=2)
X_t = sampler.fit_transform(X)
print(f"  Done in {time.time()-t0:.1f}s. Shape: {X.shape} -> {X_t.shape}  ({X_t.nbytes/1024/1024:.0f} MB)")

t1 = time.time()
print("\nStep 2: OvR LinearSVC fit (n_jobs=-1)...")
svm = OneVsRestClassifier(
    LinearSVC(C=1.0, dual='auto', max_iter=5000, random_state=42),
    n_jobs=-1,
)
svm.fit(X_t, y)
print(f"  Done in {time.time()-t1:.1f}s")
print(f"  Classes seen: {svm.classes_.tolist()}")
print(f"Total elapsed: {time.time()-t0:.1f}s")

## 4. Guardar artifact + envolver en ClassicalSVM (formato del proyecto)

Primero un artifact dict portable, luego `import_colab_svm.py` lo envuelve y produce `data/processed/classical_svm.pkl`.

In [ ]:
target_class_ids = list(range(1, 21))  # 20 clases (orden de classes.yaml)
artifact = {
    'sampler': sampler,
    'svm': svm,
    'target_class_ids': target_class_ids,
    'sklearn_version': sklearn.__version__,
}
artifact_path = "data/processed/classical_svm_artifact.pkl"
with open(artifact_path, "wb") as f:
    pickle.dump(artifact, f)
size_mb = os.path.getsize(artifact_path) / 1024 / 1024
print(f"Artifact: {artifact_path}  ({size_mb:.1f} MB)")

rc = run_script("scripts/import_colab_svm.py")
assert rc == 0, "import_colab_svm.py falló"

## 5. Subir el SVM final a Drive

In [ ]:
sync_to_drive(["classical_svm.pkl"])

## Siguiente paso

Abre `notebooks/colab_hard_neg.ipynb` para hacer hard negative mining + refit. Si te saltas esa etapa, ya puedes ir directo a `notebooks/colab_infer.ipynb`.